In [1]:
# Standard library imports
import os
import sys
import time
import random
import logging
import warnings

# Third-party imports
import torch
import optuna
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from optuna.samplers import TPESampler
from torch.utils.data import DataLoader
from optuna.pruners import MedianPruner
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit

# Local imports
module_path = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from paths import BASE_INPUT_PATH, BASE_OUTPUT_PATH

warnings.filterwarnings("ignore")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [ ]:
# Set environment variable for Python hash seed
os.environ['PYTHONHASHSEED'] = '42'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

# Set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if using multiple GPUs
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

# Worker initialization function to set seed for worker processes
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Create a generator with a fixed seed
g = torch.Generator()
g.manual_seed(seed)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

In [3]:
# Set global matplotlib parameters
plt.style.use('default')
plt.rcParams['figure.figsize'] = (100, 10)
plt.rcParams['font.size'] = 14
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# Additional parameters for better visibility on the background
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['grid.color'] = '#cccccc'
plt.rcParams['text.color'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['axes.grid'] = True

In [ ]:
# List of historical time steps to use as input features
# For example, 42 means using data from the past 42 products' values (past 7 days) to predict the next value
LOOKBACKS = [42, 168]
PRODUCT_SIGNS = ['NEG', 'POS']
PRICE_TYPES = ['AVERAGE', 'MARGINAL']

# Create output folder if it doesn't exist
output_folder = BASE_OUTPUT_PATH / 'bidirectional_lstm_model_output'
output_folder.mkdir(parents=True, exist_ok=True)

# Check if dataset exists before reading
dataset_path = BASE_INPUT_PATH / 'processed_afrr_data.csv'
if not dataset_path.exists():
    print(f"Dataset not found at {dataset_path}")
    exit(1)

# Read the dataset
try:
    processed_afrr_data = pd.read_csv(dataset_path, parse_dates=['DATE'], index_col=['DATE'])
except Exception as e:
    print(f"Error reading dataset: {e}")
    exit(1)

for lookback in LOOKBACKS:
    # Create a list to store the rows for the new DataFrame
    all_models_output = []  
    for product_sign in PRODUCT_SIGNS:
        for price_type in PRICE_TYPES:
            total_start_time = time.time()


            def preprocess_dataset(dataset, product_sign, price_type, lookback):
                price_column = f'{product_sign}_GERMANY_{price_type}_CAPACITY_PRICE_[(EUR/MW)/h]'    
                price_df = pd.DataFrame(dataset[price_column])

                # Apply log transformation - adding a small constant to handle zeros or small values
                price_df['log_price'] = np.log1p(price_df[price_column])

                # Create lag features using the log-transformed values
                lag_cols = [f'lag_{lag}' for lag in range(1, lookback + 1)]
                for lag, col in enumerate(lag_cols, 1):
                    price_df[col] = price_df['log_price'].shift(lag)

                # Scale the features to (0,1)
                scaled_columns = ['log_price'] + lag_cols
                scaler = MinMaxScaler(feature_range=(0, 1))
                price_df[scaled_columns] = scaler.fit_transform(price_df[scaled_columns])

                # Drop the original price column and reverse column order for model input
                price_df.drop(columns=[price_column], inplace=True)
                price_df = price_df.loc[:, ::-1]
                
                # Drop rows with NaN values (from the lag operations)
                price_df.dropna(inplace=True)
                
                # Convert to float32 for memory efficiency and performance
                price_df = price_df.astype(np.float32)
                return price_df, scaler, price_column

            output_path = output_folder / f'ts{lookback}'
            output_path.mkdir(parents=True, exist_ok=True)

            dataset = processed_afrr_data
            price_df, scaler, price_column = preprocess_dataset(dataset, product_sign, price_type, lookback)

            price_df_init = dataset.copy()
            price_df_init = pd.DataFrame(price_df_init[price_column])
            price_df_init = price_df_init.loc[price_df.index]

            price_df = price_df.to_numpy()

            X = price_df[:, :-1]
            y = price_df[:, -1]

            train_limit = int(len(X) * 0.7)
            validation_limit = int(len(X) * 0.85)

            X_cv_init, y_cv_init = X[:validation_limit], y[:validation_limit]

            X_train, y_train = X[:train_limit], y[:train_limit]
            X_val, y_val = X[train_limit:validation_limit], y[train_limit:validation_limit]
            X_test, y_test = X[validation_limit:], y[validation_limit:]

            X_train = X_train.reshape(-1, lookback, 1)
            X_val = X_val.reshape(-1, lookback, 1)
            X_test = X_test.reshape(-1, lookback, 1)

            y_train = y_train.reshape(-1, 1)
            y_val = y_val.reshape(-1, 1)
            y_test = y_test.reshape(-1, 1)

            X_train = torch.tensor(X_train).float()
            X_val = torch.tensor(X_val).float()
            X_test = torch.tensor(X_test).float()

            y_train = torch.tensor(y_train).float()
            y_val = torch.tensor(y_val).float()
            y_test = torch.tensor(y_test).float()


            class TimeSeriesDataset(Dataset):
                def __init__(self, X, y):
                    self.X = X
                    self.y = y

                def __len__(self):
                    return len(self.X)

                def __getitem__(self, i):
                    return self.X[i], self.y[i]
                
            train_dataset = TimeSeriesDataset(X_train, y_train)
            val_dataset = TimeSeriesDataset(X_val, y_val)
            test_dataset = TimeSeriesDataset(X_test, y_test)


            def create_dataloaders(batch_size):
                # Use the same generator and worker initialization for all loaders
                train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                                        worker_init_fn=seed_worker, generator=g, 
                                        num_workers=0)
                val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                                    worker_init_fn=seed_worker, generator=g,
                                    num_workers=0)
                test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                                        worker_init_fn=seed_worker, generator=g,
                                        num_workers=0)
                return train_loader, val_loader, test_loader


            # Define the BiLSTM model with Dropout
            class BiLSTM(nn.Module):
                def __init__(self, input_size, hidden_size, num_stacked_layers, dropout_rate):
                    super().__init__()
                    self.hidden_size = hidden_size
                    self.num_stacked_layers = num_stacked_layers
                    self.lstm = nn.LSTM(input_size, hidden_size, num_stacked_layers, 
                                        batch_first=True, dropout=dropout_rate, bidirectional=True)
                    # Note: For bidirectional LSTM, the output size is doubled
                    self.fc = nn.Linear(hidden_size * 2, 1)
                    self.dropout = nn.Dropout(dropout_rate)

                def forward(self, x):
                    batch_size = x.size(0)
                    # Initialize hidden state with zeros for both directions (×2)
                    h0 = torch.zeros(self.num_stacked_layers * 2, batch_size, self.hidden_size).to(device)
                    c0 = torch.zeros(self.num_stacked_layers * 2, batch_size, self.hidden_size).to(device)
                    # Forward pass
                    out, _ = self.lstm(x, (h0, c0))
                    out = self.dropout(out[:, -1, :])
                    out = self.fc(out)
                    return out
                

            def time_series_cv(X, y, model_class, model_params, n_splits=5, batch_size=64, 
                            learning_rate=0.001, epochs=50, patience=10, device='cuda'):
                # Initialize TimeSeriesSplit
                tscv = TimeSeriesSplit(n_splits=n_splits)
                
                cv_scores = []
                cv_models = []
                fold = 1
                
                for train_idx, val_idx in tscv.split(X):
                    print(f"Training fold {fold}/{n_splits}")
                    
                    # Split data
                    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
                    y_train_fold, y_val_fold = y[train_idx], y[val_idx]
                    
                    # Convert to torch tensors
                    X_train_fold = torch.tensor(X_train_fold).float()
                    y_train_fold = torch.tensor(y_train_fold).float()
                    X_val_fold = torch.tensor(X_val_fold).float()
                    y_val_fold = torch.tensor(y_val_fold).float()
                    
                    # Create datasets and loaders
                    train_dataset = TimeSeriesDataset(X_train_fold, y_train_fold)
                    val_dataset = TimeSeriesDataset(X_val_fold, y_val_fold)
                    
                    # Use your seed_worker and generator g for reproducibility
                    train_loader = DataLoader(
                        train_dataset, batch_size=batch_size, shuffle=True,
                        worker_init_fn=seed_worker, generator=g, num_workers=0
                    )
                    val_loader = DataLoader(
                        val_dataset, batch_size=batch_size, shuffle=False,
                        worker_init_fn=seed_worker, generator=g, num_workers=0
                    )
                    
                    # Initialize model
                    model = model_class(**model_params).to(device)
                    
                    # Loss and optimizer
                    # There are several options for loss functions (e.g. nn.L1Loss(), nn.CrossEntropyLoss(), nn.HuberLoss()), 
                    # but MSE is the default and a good starting point
                    loss_function = nn.MSELoss()
                    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
                    
                    # Training with early stopping
                    best_val_loss = float('inf')
                    counter = 0
                    best_model_state = None
                    
                    for epoch in range(epochs):
                        # Training
                        model.train()
                        running_loss = 0.0
                        for X_batch, y_batch in train_loader:
                            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                            optimizer.zero_grad()
                            output = model(X_batch)
                            loss = loss_function(output, y_batch)
                            loss.backward()
                            optimizer.step()
                            running_loss += loss.item()
                        
                        # Validation
                        model.eval()
                        val_loss = 0.0
                        with torch.no_grad():
                            for X_batch, y_batch in val_loader:
                                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                                output = model(X_batch)
                                loss = loss_function(output, y_batch)
                                val_loss += loss.item()
                        
                        val_loss = val_loss / len(val_loader)
                        
                        # Print progress every few epochs
                        if epoch % 10 == 0:
                            print(f"Fold {fold}, Epoch {epoch+1}/{epochs}, Val Loss: {val_loss:.6f}")
                        
                        # Early stopping
                        if val_loss < best_val_loss:
                            best_val_loss = val_loss
                            counter = 0
                            best_model_state = model.state_dict().copy()
                        else:
                            counter += 1
                            if counter >= patience:
                                print(f"Early stopping at epoch {epoch+1}")
                                break
                    
                    # Load best model
                    model.load_state_dict(best_model_state)
                    
                    # Final validation score
                    model.eval()
                    final_val_loss = 0.0
                    with torch.no_grad():
                        for X_batch, y_batch in val_loader:
                            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                            output = model(X_batch)
                            loss = loss_function(output, y_batch)
                            final_val_loss += loss.item()
                    
                    final_val_loss = final_val_loss / len(val_loader)
                    cv_scores.append(final_val_loss)
                    cv_models.append(model)
                    
                    print(f"Fold {fold} validation loss: {final_val_loss:.6f}")
                    fold += 1
                
                return cv_scores, cv_models


            def visualize_ts_cv_splits(X, index, n_splits=5):
                tscv = TimeSeriesSplit(n_splits=n_splits)
                
                plt.figure(figsize=(15, 8))
                for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
                    fold_train_dates = index[train_idx]
                    fold_test_dates = index[test_idx]
                    
                    plt.scatter(fold_train_dates, [i+0.1]*len(fold_train_dates), 
                            c='blue', s=10, label='Train' if i == 0 else "")
                    plt.scatter(fold_test_dates, [i+0.2]*len(fold_test_dates), 
                            c='red', s=10, label='Validation' if i == 0 else "")
                
                plt.yticks(np.arange(0.15, n_splits+0.15, 1), [f"Fold {i+1}" for i in range(n_splits)])
                plt.title('Time Series Cross-Validation Splits')
                plt.ylabel('CV Iteration')
                plt.xlabel('Date')
                plt.legend()
                plt.tight_layout()
                plt.show()


            def create_ensemble(cv_models, X_test, device='cuda'):
                X_test_tensor = torch.tensor(X_test).float().to(device)
                
                predictions = []
                for model in cv_models:
                    model.eval()
                    with torch.no_grad():
                        pred = model(X_test_tensor)
                        predictions.append(pred.cpu().numpy())
                
                # Average the predictions
                ensemble_pred = np.mean(predictions, axis=0)
                return ensemble_pred


            def create_ensemble_recency_weighted(cv_models, X_test, device='cuda', recency_factor=1.5):
                X_test_tensor = torch.tensor(X_test).float().to(device)
                
                # Generate weights based on recency
                # More recent models (higher index in the list) get higher weights
                num_models = len(cv_models)
                weights = np.array([(i+1)**recency_factor for i in range(num_models)])
                weights = weights / weights.sum()  # Normalize to sum to 1
                                
                # Get predictions from all models
                predictions = []
                for model in cv_models:
                    model.eval()
                    with torch.no_grad():
                        pred = model(X_test_tensor)
                        predictions.append(pred.cpu().numpy())
                
                # Apply weights to each model's predictions
                weighted_predictions = np.zeros_like(predictions[0])
                for i, pred in enumerate(predictions):
                    weighted_predictions += weights[i] * pred
                
                return weighted_predictions


            # Phase 1: Optimize architecture parameters
            def phase1_objective(trial):
                # Architecture parameters
                hidden_size = trial.suggest_int('hidden_size', 64, 128)
                num_stacked_layers = trial.suggest_int('num_stacked_layers', 1, 2)
                
                # Fixed parameters for phase 1
                learning_rate = 0.001  # Fixed medium learning rate
                dropout_rate = 0.2     # Fixed medium dropout
                batch_size = 64        # Fixed batch size
                
                # Print trial info
                print(f"Phase 1 - Trial #{trial.number}")
                
                # Create the model
                model = BiLSTM(input_size=1, hidden_size=hidden_size, 
                            num_stacked_layers=num_stacked_layers, 
                            dropout_rate=dropout_rate).to(device)
                
                # Training setup
                train_loader, val_loader, _ = create_dataloaders(batch_size)
                loss_function = nn.MSELoss()
                optimizer = optim.Adam(model.parameters(), lr=learning_rate)
                
                # Training with early stopping
                return train_and_evaluate(trial, model, optimizer, loss_function, train_loader, val_loader)


            # Phase 2: Optimize regularization parameters with best architecture
            def phase2_objective(trial):
                # Use best architecture from phase 1
                hidden_size = phase1_best_params['hidden_size']
                num_stacked_layers = phase1_best_params['num_stacked_layers']
                
                # Regularization parameters
                dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.3)
                weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)
                
                # Fixed parameters for phase 2
                learning_rate = 0.001  # Fixed medium learning rate
                batch_size = 64        # Fixed batch size
                
                # Print trial info
                print(f"Phase 2 - Trial #{trial.number}")
                
                # Create the model
                model = BiLSTM(input_size=1, hidden_size=hidden_size, 
                            num_stacked_layers=num_stacked_layers, 
                            dropout_rate=dropout_rate).to(device)
                
                # Training setup
                train_loader, val_loader, _ = create_dataloaders(batch_size)
                loss_function = nn.MSELoss()
                optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
                
                # Training with early stopping
                return train_and_evaluate(trial, model, optimizer, loss_function, train_loader, val_loader)


            # Phase 3: Optimize learning parameters with best architecture and regularization
            def phase3_objective(trial):
                # Use best architecture from phase 1
                hidden_size = phase1_best_params['hidden_size']
                num_stacked_layers = phase1_best_params['num_stacked_layers']
                
                # Use best regularization from phase 2
                dropout_rate = phase2_best_params['dropout_rate']
                weight_decay = phase2_best_params['weight_decay']
                
                # Learning parameters
                learning_rate = trial.suggest_float('learning_rate', 5e-4, 1e-3, log=True)
                batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
                
                # Print trial info
                print(f"Phase 3 - Trial #{trial.number}")
                
                # Create the model
                model = BiLSTM(input_size=1, hidden_size=hidden_size, 
                            num_stacked_layers=num_stacked_layers, 
                            dropout_rate=dropout_rate).to(device)
                
                # Training setup
                train_loader, val_loader, _ = create_dataloaders(batch_size)
                loss_function = nn.MSELoss()
                optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
                
                # Training with early stopping
                return train_and_evaluate(trial, model, optimizer, loss_function, train_loader, val_loader)

            # Common training function used by all phases
            def train_and_evaluate(trial, model, optimizer, loss_function, train_loader, val_loader, num_epochs=20, patience=5, seed=42):
                start_time = time.time()
                
                best_val_loss = float('inf')
                counter = 0
                
                for epoch in range(num_epochs):
                    # Training
                    torch.manual_seed(seed + epoch)
                    model.train()
                    running_loss = 0.0
                    
                    for batch_idx, batch in enumerate(train_loader):
                        X_batch, y_batch = batch[0].to(device), batch[1].to(device)
                        optimizer.zero_grad()
                        output = model(X_batch)
                        loss = loss_function(output, y_batch)
                        loss.backward()
                        optimizer.step()
                        running_loss += loss.item()
                    
                    # Validation
                    torch.manual_seed(seed)
                    model.eval()
                    val_loss = 0.0
                    with torch.no_grad():
                        for batch in val_loader:
                            X_batch, y_batch = batch[0].to(device), batch[1].to(device)
                            output = model(X_batch)
                            loss = loss_function(output, y_batch)
                            val_loss += loss.item()
                    
                    val_loss = val_loss / len(val_loader)
                    
                    # Report to Optuna for pruning
                    trial.report(val_loss, epoch)
                    if trial.should_prune():
                        raise optuna.exceptions.TrialPruned()
                    
                    # Early stopping
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        counter = 0
                    else:
                        counter += 1
                        if counter >= patience:
                            print(f"Early stopping at epoch {epoch+1}")
                            break
                
                execution_time = time.time() - start_time
                print(f"Training completed in {execution_time:.2f} seconds with val_loss: {best_val_loss:.6f}")
                
                return best_val_loss


            # Main optimization process
            def optimize_lstm_hyperparameters(n_trials_per_phase=20):
                global phase1_best_params, phase2_best_params
                
                print("Starting Phase 1: Architecture Optimization")
                phase1_study = optuna.create_study(
                    direction="minimize",
                    sampler=TPESampler(seed=42),
                    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
                    study_name="phase1_architecture"
                )
                phase1_study.optimize(phase1_objective, n_trials=n_trials_per_phase)
                phase1_best_params = phase1_study.best_params
                
                print("\nStarting Phase 2: Regularization Optimization")
                phase2_study = optuna.create_study(
                    direction="minimize",
                    sampler=TPESampler(seed=42),
                    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
                    study_name="phase2_regularization"
                )
                phase2_study.optimize(phase2_objective, n_trials=n_trials_per_phase)
                phase2_best_params = phase2_study.best_params
                
                print("\nStarting Phase 3: Learning Parameters Optimization")
                phase3_study = optuna.create_study(
                    direction="minimize",
                    sampler=TPESampler(seed=42),
                    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
                    study_name="phase3_learning"
                )
                phase3_study.optimize(phase3_objective, n_trials=n_trials_per_phase)
                phase3_best_params = phase3_study.best_params
                
                # Combine all best parameters
                best_params = {
                    **phase1_best_params,
                    **phase2_best_params,
                    **phase3_best_params
                }
                return best_params


            # Initialize global variables to store best parameters between phases
            phase1_best_params = {}
            phase2_best_params = {}
            phase3_best_params = {}

            # Run the optimization process
            best_params = optimize_lstm_hyperparameters(n_trials_per_phase=20)

            # Define model parameters from your best hyperparameter search
            model_params = {
                'input_size': 1, 
                'hidden_size': best_params['hidden_size'],
                'num_stacked_layers': best_params['num_stacked_layers'],
                'dropout_rate': best_params['dropout_rate']
            }

            # Define X and y for cross-validation (using all available data)
            X_cv = X_cv_init.reshape(-1, lookback, 1)
            y_cv = y_cv_init.reshape(-1, 1)

            # Visualize the CV splits first
            visualize_ts_cv_splits(X_cv_init, price_df_init.index, n_splits=5)

            # Run time series cross-validation
            cv_scores, cv_models = time_series_cv(
                X=X_cv,
                y=y_cv,
                model_class=BiLSTM,
                model_params=model_params,
                n_splits=5,
                batch_size=best_params.get('batch_size', 64),
                learning_rate=best_params.get('learning_rate', 0.001),
                epochs=100,
                patience=15,
                device=device
            )

            # Create ensemble and recency-weighted predictions for validation data
            ensemble_val_preds = create_ensemble(cv_models, X_val, device=device)
            recency_weighted_val_preds = create_ensemble_recency_weighted(cv_models, X_val, device=device, recency_factor=1.5)

            # Convert predictions and actual values back to original scale
            # First, create dummy arrays with zeros for all features
            y_val_np = y_val.cpu().numpy()
            dummy_val = np.zeros((len(y_val_np), lookback + 1))
            dummy_pred = np.zeros((len(ensemble_val_preds), lookback + 1))
            dummy_recency_pred = np.zeros((len(recency_weighted_val_preds), lookback + 1))

            # Put the scaled values in the last column (target column)
            dummy_val[:, -1] = y_val_np.flatten()
            dummy_pred[:, -1] = ensemble_val_preds.flatten()
            dummy_recency_pred[:, -1] = recency_weighted_val_preds.flatten()

            # Inverse transform
            y_val_inv = scaler.inverse_transform(dummy_val)[:, -1]                  # Get just the target column
            pred_inv = scaler.inverse_transform(dummy_pred)[:, -1]                  # Get just the target column
            recency_pred_inv = scaler.inverse_transform(dummy_recency_pred)[:, -1]  # Get just the target column

            # Apply inverse log transform (expm1 is inverse of log1p)
            y_val_original = np.expm1(y_val_inv)
            ensemble_val_pred_original = np.expm1(pred_inv)
            recency_val_pred_original = np.expm1(recency_pred_inv)

            # Create ensemble and recency-weighted predictions for test data
            ensemble_test_preds = create_ensemble(cv_models, X_test, device=device)
            recency_weighted_test_preds = create_ensemble_recency_weighted(cv_models, X_test, device=device, recency_factor=1.5)

            # Convert predictions and actual values back to original scale
            # First, create dummy arrays with zeros for all features
            y_test_np = y_test.cpu().numpy()
            dummy_test = np.zeros((len(y_test_np), lookback + 1))
            dummy_pred = np.zeros((len(ensemble_test_preds), lookback + 1))
            dummy_recency_pred = np.zeros((len(recency_weighted_test_preds), lookback + 1))

            # Put the scaled values in the last column (target column)
            dummy_test[:, -1] = y_test_np.flatten()
            dummy_pred[:, -1] = ensemble_test_preds.flatten()
            dummy_recency_pred[:, -1] = recency_weighted_test_preds.flatten()

            # Inverse transform
            y_test_inv = scaler.inverse_transform(dummy_test)[:, -1]                # Get just the target column
            pred_inv = scaler.inverse_transform(dummy_pred)[:, -1]                  # Get just the target column
            recency_pred_inv = scaler.inverse_transform(dummy_recency_pred)[:, -1]  # Get just the target column

            # Apply inverse log transform (expm1 is inverse of log1p)
            y_test_original = np.expm1(y_test_inv)
            ensemble_test_pred_original = np.expm1(pred_inv)
            recency_test_pred_original = np.expm1(recency_pred_inv)

            # Train a single model with the best parameters on all training data
            best_model = BiLSTM(**model_params).to(device)
            loss_function = nn.MSELoss()
            optimizer = torch.optim.Adam(best_model.parameters(), lr=best_params.get('learning_rate', 0.001))

            # Create data loaders
            train_loader, val_loader, test_loader = create_dataloaders(batch_size=best_params.get('batch_size', 64))

            # Train the model
            best_val_loss = float('inf')
            epochs = 100
            patience = 15
            counter = 0

            train_losses = []
            val_losses = []

            for epoch in range(epochs):
                # Training
                best_model.train()
                running_loss = 0.0
                for X_batch, y_batch in train_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    optimizer.zero_grad()
                    output = best_model(X_batch)
                    loss = loss_function(output, y_batch)
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item()
                
                train_loss = running_loss/len(train_loader)
                train_losses.append(train_loss)
                
                # Validation
                best_model.eval()
                val_loss = 0.0
                with torch.no_grad():
                    for X_batch, y_batch in val_loader:
                        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                        output = best_model(X_batch)
                        loss = loss_function(output, y_batch)
                        val_loss += loss.item()
                
                val_loss = val_loss / len(val_loader)
                val_losses.append(val_loss)

                # Print progress every 10 epochs
                if epoch % 10 == 0:
                    print(f"Epoch {epoch+1}/{epochs}, Val Loss: {val_loss:.6f}")
                
                # Early stopping
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    counter = 0
                    torch.save(best_model.state_dict(), "best_lstm_model.pth")
                else:
                    counter += 1
                    if counter >= patience:
                        print(f"Early stopping at epoch {epoch+1}")
                        break

            # Load best model
            best_model.load_state_dict(torch.load("best_lstm_model.pth"))

            # Create a figure for the loss plot
            plt.figure(figsize=(12, 6))
            plt.plot(train_losses, label='Training Loss')
            plt.plot(val_losses, label='Validation Loss')
            plt.xlabel('Epoch')
            plt.ylabel('Loss (MSE)')
            plt.title(f'Training and Validation Loss for {product_sign} {price_type}')
            plt.legend()
            plt.grid(True)
            plt.tight_layout()

            # Optional: Add markers to better visualize the best epoch
            best_epoch = np.argmin(val_losses)
            plt.scatter(best_epoch, val_losses[best_epoch], marker='o', color='red', s=100, 
                        label=f'Best Model (Epoch {best_epoch+1})')
            plt.legend()
            plt.tight_layout()
            plt.savefig(output_path / f'{product_sign}_{price_type}_training_validation_loss.png')
            plt.close()

            # Validation single best model
            best_model.eval()
            single_val_preds = []

            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch = X_batch.to(device)
                    outputs = best_model(X_batch)
                    single_val_preds.extend(outputs.cpu().numpy())

            single_val_preds = np.array(single_val_preds)

            # Convert single model predictions back to original scale
            dummy_pred = np.zeros((len(single_val_preds), lookback + 1))

            # Put the scaled values in the last column (target column)
            dummy_pred[:, -1] = single_val_preds.flatten()

            # Inverse transform
            pred_inv = scaler.inverse_transform(dummy_pred)[:, -1]     # Get just the target column

            # Apply inverse log transform (expm1 is inverse of log1p)
            single_val_pred_original = np.expm1(pred_inv)

            # Test single best model
            best_model.eval()
            single_test_preds = []

            with torch.no_grad():
                for X_batch, y_batch in test_loader:
                    X_batch = X_batch.to(device)
                    outputs = best_model(X_batch)
                    single_test_preds.extend(outputs.cpu().numpy())

            single_test_preds = np.array(single_test_preds)

            # Convert single model predictions back to original scale
            dummy_pred = np.zeros((len(single_test_preds), lookback + 1))

            # Put the scaled values in the last column (target column)
            dummy_pred[:, -1] = single_test_preds.flatten()

            # Inverse transform
            pred_inv = scaler.inverse_transform(dummy_pred)[:, -1]     # Get just the target column

            # Apply inverse log transform (expm1 is inverse of log1p)
            single_pred_test_original = np.expm1(pred_inv)

            total_execution_time = time.time() - total_start_time

            df_val = pd.DataFrame({
                'ACTUAL_VALUE': y_val_original,
                'D+1_ENSEMBLE_MODEL': ensemble_val_pred_original,
                'D+1_RECENCY_WEIGHTED': recency_val_pred_original, 
                'D+1_SINGLE_MODEL': single_val_pred_original
            })

            df_test = pd.DataFrame({
                'ACTUAL_VALUE': y_test_original,
                'D+1_ENSEMBLE_MODEL': ensemble_test_pred_original,
                'D+1_RECENCY_WEIGHTED': recency_test_pred_original,
                'D+1_SINGLE_MODEL': single_pred_test_original
            })


            # Function to determine product based on hour
            def determine_product(hour):
                if hour == 0:
                    return '00_04'
                elif hour == 4:
                    return '04_08'
                elif hour == 8:
                    return '08_12'
                elif hour == 12:
                    return '12_16'
                elif hour == 16:
                    return '16_20'
                elif hour == 20:
                    return '20_24'
                else:
                    return 'all'
                
            
            # Define val_indices and test_indices based on the price_df_init index slicing
            val_indices = price_df_init[train_limit:validation_limit].index
            test_indices = price_df_init[validation_limit:].index

            # Set proper datetime indices for validation and test DataFrames
            df_val.index = pd.DatetimeIndex(val_indices)
            df_test.index = pd.DatetimeIndex(test_indices)

            # Apply the function to create the 'product' column
            df_val['PRODUCT'] = df_val.index.hour.map(determine_product)
            df_test['PRODUCT'] = df_test.index.hour.map(determine_product)

            # Separate the values into 6 different DataFrames relative to the values of the column 'product'
            products = ['00_04', '04_08', '08_12', '12_16', '16_20', '20_24']

            dfs_val = {product: df_val[df_val['PRODUCT'] == product] for product in products}
            dfs_test = {product: df_test[df_test['PRODUCT'] == product] for product in products}

            # Combine validation and test DataFrames
            dfs_combined = {product: pd.concat([dfs_val[product], dfs_test[product]]) for product in products}

            # Remove PRODUCT column from each DataFrame since it's redundant
            for product in products:
                dfs_combined[product] = dfs_combined[product].drop(columns=['PRODUCT'])
                file_name = f'bidirectional_lstm_model_results_{product}_{product_sign}_{price_type}.csv'
                file_path = output_path / file_name
                dfs_combined[product].to_csv(file_path)

            # Add recency metrics to the new row
            new_row = {
                'PRODUCT': 'all',
                'PRODUCT_SIGN': product_sign,
                'PRICE_TYPE': price_type,
                'TIMESTEP': lookback,
                'HIDDEN_SIZE': best_params['hidden_size'],
                'NUM_STACKED_LAYERS': best_params['num_stacked_layers'],
                'LEARNING_RATE': best_params['learning_rate'],
                'DROPOUT_RATE': best_params['dropout_rate'],
                'WEIGHT_DECAY': best_params['weight_decay'],
                'BATCH_SIZE': best_params['batch_size'],
                'EXECUTION_TIME_[S]': total_execution_time,
            }
            
            # Append the new row to the list
            all_models_output.append(new_row)

    # Convert the list to a DataFrame
    all_models_output_df = pd.DataFrame(all_models_output)

    # Save the DataFrame to a CSV file
    output_file_path = output_path / f'bidirectional_lstm_model_overview_ts{lookback}.csv'
    all_models_output_df.to_csv(output_file_path, index=False)